Read files from cloud storage by datastream reader

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType,
    TimestampType
)

cust_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("member_since", DateType(), True),
    StructField("created_timestamp", TimestampType(), True),
])

In [0]:
customer_df = (spark.readStream
                   .format("json")
                   .schema(cust_schema)
                   .load("/Volumes/project_etl/landing/opertaional/customer_streams/")
                #    .option("header", "true")
                   )

# display(customer_df);

In [0]:
from pyspark.sql.functions import col, current_timestamp
cust_df_add = (customer_df
                #    .withColumn("filepath", col("_metadata.filepath"))
                #    .withColumn("file_name", col("_metadata.file_name"))
                   .withColumn("loading_ts", current_timestamp()))

In [0]:
cust_df_add.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/project_etl/landing/opertaional/customer_streams/_checkpoint_stream_customer").trigger(availableNow=True)\
    .toTable("project_etl.bronze.customer_stream")

In [0]:
%sql
select * from project_etl.bronze.customer_stream